# 05 — Weekly Injury Adjustment

This notebook adds a small, player sensitive injury layer to the weekly model.

The goal is not to treat every injury as a major team strength event. NFL rosters contain replacement level talent, and the next player up can often absorb a meaningful portion of the missing player's role.

The adjustment therefore depends on:

1. **Expected availability** from the current injury/practice report.
2. **Player role** from the latest depth chart.
3. **Position importance**, with quarterback handled separately.
4. **Replacement quality**, represented conservatively by depth chart structure.
5. **A hard cap** so injuries cannot overwhelm the underlying team strength model.

This is intentionally a secondary adjustment. Preseason strength and observed team performance remain the primary drivers.

The original preseason notebooks and ratings are never modified.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import nflreadpy as nfl


In [2]:
PROJECT_ROOT = Path("../..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 3

TEAM_STRENGTH_PATH = (
    WEEKLY_DATA_DIR / f"week_{TARGET_WEEK:02d}_team_strength.parquet"
)

OUTPUT_PATH = (
    WEEKLY_DATA_DIR / f"week_{TARGET_WEEK:02d}_injury_adjusted_team_strength.parquet"
)

INJURY_DETAIL_PATH = (
    WEEKLY_DATA_DIR / f"week_{TARGET_WEEK:02d}_injury_adjustments.parquet"
)


## Load Current Injury Reports and Depth Charts

`nflreadpy` is used so this can be rerun as the official injury report changes during the week.

**Important:** early in the week, official game status designations may not yet be available. In that case, the notebook uses practice participation conservatively and prints a warning. Rerun this notebook closer to kickoff as reports become more complete.


In [3]:
team_strength = pd.read_parquet(TEAM_STRENGTH_PATH)

injuries = nfl.load_injuries([SEASON]).to_pandas()
depth = nfl.load_depth_charts([SEASON]).to_pandas()

injuries = injuries[
    (injuries["season"] == SEASON)
    & (injuries["week"] == TARGET_WEEK)
].copy()

print("Week injury rows:", len(injuries))
print(
    "Teams represented:",
    injuries["team"].nunique() if len(injuries) else 0
)

if len(injuries):
    print(f"Week {TARGET_WEEK} injury data is available.")
else:
    print(f"No Week {TARGET_WEEK} injury data available yet.")

Week injury rows: 259
Teams represented: 30
Week 3 injury data is available.


## Keep the Latest Depth Chart

Depth chart rank is used as a player specific role proxy.

A starter matters more than a reserve, but the model deliberately does not assume the starter's full value disappears when he is limited or unavailable.


In [4]:
if "dt" in depth.columns:
    depth["dt"] = pd.to_datetime(depth["dt"], errors="coerce")
    latest_dt = depth["dt"].max()
    latest_depth = depth[depth["dt"] == latest_dt].copy()
else:
    latest_depth = depth.copy()
    latest_dt = None

depth_cols = [
    c for c in [
        "team", "player_name", "gsis_id", "pos_grp",
        "pos_name", "pos_abb", "pos_slot", "pos_rank"
    ]
    if c in latest_depth.columns
]

latest_depth = latest_depth[depth_cols].copy()

print("Latest depth-chart date:", latest_dt)
print("Depth-chart rows:", len(latest_depth))


Latest depth-chart date: 2026-09-24 12:42:08+00:00
Depth-chart rows: 2228


## Availability / Severity Translation

The injury type itself is retained for diagnostics, but we do not assign made up point values to labels such as ankle, hamstring, shoulder, or knee.

Instead, severity enters through the team's official participation and game status information.

The values below represent expected lost availability, not expected lost team value:

- Out / IR-like designation: 100%
- Doubtful: 80%
- Questionable: 35%
- Full practice / no game designation: 0%
- Limited practice without a designation: 10%
- Did not practice without a designation: 25%

These are deliberately conservative and can later be replaced with probabilities estimated from historical injury report to snap outcomes.


In [5]:
def normalize_text(value):
    if pd.isna(value):
        return ""
    return str(value).strip().lower()

def expected_lost_availability(row):
    report_status = str(row.get("report_status", "")).lower()
    practice_status = str(row.get("practice_status", "")).lower()

    # Official game designation takes priority
    if "out" in report_status or "reserve" in report_status or "ir" in report_status:
        return 1.00

    if "doubtful" in report_status:
        return 0.80

    if "questionable" in report_status:
        return 0.35

    # If there is not yet a game designation,
    # use practice participation conservatively
    if "did not participate" in practice_status:
        return 0.25

    if "limited participation" in practice_status:
        return 0.10

    if "full participation" in practice_status:
        return 0.00

    return 0.00

if len(injuries):
    injuries["expected_lost_availability"] = injuries.apply(
        expected_lost_availability,
        axis=1
    )


## Match Injured Players to Their Current Role

GSIS ID is preferred because names can differ across sources. Name matching is used only as a fallback.


In [6]:
injury_detail = injuries.copy()

if len(injury_detail):
    if "gsis_id" in latest_depth.columns and "gsis_id" in injury_detail.columns:
        depth_by_id = (
            latest_depth
            .dropna(subset=["gsis_id"])
            .sort_values("pos_rank")
            .drop_duplicates("gsis_id")
        )
        injury_detail = injury_detail.merge(
            depth_by_id[
                [c for c in [
                    "gsis_id", "pos_grp", "pos_abb",
                    "pos_slot", "pos_rank"
                ] if c in depth_by_id.columns]
            ],
            on="gsis_id",
            how="left"
        )

    if "pos_rank" not in injury_detail.columns:
        injury_detail["pos_rank"] = np.nan
    if "pos_abb" not in injury_detail.columns:
        injury_detail["pos_abb"] = injury_detail.get("position", "")
    else:
        injury_detail["pos_abb"] = injury_detail["pos_abb"].fillna(
            injury_detail.get("position")
        )

    injury_detail["pos_rank"] = pd.to_numeric(
        injury_detail["pos_rank"], errors="coerce"
    ).fillna(3)


## Conservative Player Importance

This is where the "next man up" principle enters.

Depth chart role multipliers:

- Rank 1: 1.00
- Rank 2: 0.45
- Rank 3+: 0.20

The maximum effect is also position specific. Quarterback has the largest possible impact, while every non QB position is capped well below that.

These are caps, not automatic deductions. The final deduction is:

`availability loss × role multiplier × position cap × replacement discount`

The replacement discount is 0.60, meaning the model assumes a meaningful share of the player's role can be replaced rather than treating his absence as a complete loss.


In [7]:
ROLE_MULTIPLIER = {
    1: 1.00,
    2: 0.45
}

POSITION_POINT_CAP = {
    "QB": 4.00,
    "OL": 0.70,
    "T": 0.70,
    "OT": 0.70,
    "G": 0.55,
    "OG": 0.55,
    "C": 0.55,
    "WR": 0.65,
    "TE": 0.45,
    "RB": 0.35,
    "FB": 0.20,
    "DE": 0.60,
    "EDGE": 0.60,
    "DT": 0.45,
    "NT": 0.40,
    "LB": 0.45,
    "OLB": 0.50,
    "ILB": 0.40,
    "MLB": 0.40,
    "CB": 0.60,
    "DB": 0.50,
    "S": 0.50,
    "FS": 0.50,
    "SS": 0.50,
    "K": 0.20,
    "P": 0.10
}

REPLACEMENT_DISCOUNT = 0.60

def role_multiplier(rank):
    if rank <= 1:
        return 1.00
    if rank <= 2:
        return 0.45
    return 0.20

def position_cap(position):
    return POSITION_POINT_CAP.get(
        normalize_text(position).upper(),
        0.25
    )

if len(injury_detail):
    injury_detail["role_multiplier"] = (
        injury_detail["pos_rank"].apply(role_multiplier)
    )
    injury_detail["position_point_cap"] = (
        injury_detail["pos_abb"].apply(position_cap)
    )

    injury_detail["raw_injury_point_cost"] = (
        injury_detail["expected_lost_availability"]
        * injury_detail["role_multiplier"]
        * injury_detail["position_point_cap"]
    )

    injury_detail["injury_point_cost"] = (
        injury_detail["raw_injury_point_cost"]
        * REPLACEMENT_DISCOUNT
    )


## Team Level Injury Adjustment

Multiple injuries can accumulate, but the total team adjustment is capped.

This prevents a long injury report from mechanically destroying a team's rating and reflects the fact that NFL teams plan for depth.

A negative value means the current injury situation lowers the team's weekly rating.


In [8]:
MAX_TEAM_INJURY_PENALTY = 4.5

if len(injury_detail):
    team_injuries = (
        injury_detail
        .groupby("team", as_index=False)
        .agg(
            injury_report_players=("gsis_id", "count"),
            injury_point_cost=("injury_point_cost", "sum")
        )
    )

    team_injuries["injury_adjustment"] = -np.minimum(
        team_injuries["injury_point_cost"],
        MAX_TEAM_INJURY_PENALTY
    )
else:
    team_injuries = pd.DataFrame({
        "team": team_strength["team"],
        "injury_report_players": 0,
        "injury_point_cost": 0.0,
        "injury_adjustment": 0.0
    })

injury_adjusted = team_strength.merge(
    team_injuries,
    on="team",
    how="left"
)

injury_adjusted[
    ["injury_report_players", "injury_point_cost", "injury_adjustment"]
] = injury_adjusted[
    ["injury_report_players", "injury_point_cost", "injury_adjustment"]
].fillna(0)

injury_adjusted["pre_injury_weekly_team_strength"] = (
    injury_adjusted["weekly_team_strength"]
)

injury_adjusted["injury_adjusted_team_strength"] = (
    injury_adjusted["pre_injury_weekly_team_strength"]
    + injury_adjusted["injury_adjustment"]
)


## Diagnostics

This is an important review point. Injury adjustments should generally be small.

A major quarterback absence can matter substantially, but ordinary questionable/limited players should move the model only slightly.


In [9]:
print("TEAM INJURY ADJUSTMENTS")
display(
    injury_adjusted[
        [
            "team",
            "pre_injury_weekly_team_strength",
            "injury_report_players",
            "injury_adjustment",
            "injury_adjusted_team_strength"
        ]
    ]
    .sort_values("injury_adjustment")
    .round(3)
)

if len(injury_detail):
    print("\nLARGEST INDIVIDUAL INJURY EFFECTS")
    detail_cols = [
        c for c in [
            "team", "full_name", "position",
            "report_primary_injury", "report_status",
            "practice_status", "pos_rank",
            "expected_lost_availability",
            "injury_point_cost"
        ] if c in injury_detail.columns
    ]
    display(
        injury_detail[detail_cols]
        .sort_values("injury_point_cost", ascending=False)
        .head(25)
        .round(3)
    )


TEAM INJURY ADJUSTMENTS


,team,pre_injury_weekly_team_strength,injury_report_players,injury_adjustment,injury_adjusted_team_strength
23,WAS,-3.328,16.0,-0.952,-4.280
15,GB,0.963,12.0,-0.441,0.522
21,LAC,-1.375,14.0,-0.438,-1.813
25,NYG,-3.958,12.0,-0.422,-4.380
1,SEA,5.906,9.0,-0.402,5.504
3,LA,4.265,8.0,-0.320,3.945
2,SF,5.121,8.0,-0.312,4.809
12,HOU,1.103,11.0,-0.309,0.794
4,BAL,3.401,10.0,-0.270,3.131
22,LV,-3.241,9.0,-0.265,-3.507



LARGEST INDIVIDUAL INJURY EFFECTS


,team,full_name,position,report_primary_injury,report_status,practice_status,pos_rank,expected_lost_availability,injury_point_cost
254,WAS,Jayden Daniels,QB,NaN,NaN,Did Not Participate In Practice,1.0,0.25,0.600
199,NYG,Jaxson Dart,QB,NaN,NaN,Did Not Participate In Practice,2.0,0.25,0.270
215,SEA,Sam Darnold,QB,NaN,NaN,Limited Participation in Practice,1.0,0.10,0.240
78,GB,Jayden Reed,WR,Neck,Out,Did Not Participate In Practice,2.0,1.00,0.176
9,ATL,Samson Ebukam,DE,Hamstring,Out,Did Not Participate In Practice,1.0,1.00,0.150
148,LV,Aidan O'Connell,QB,NaN,NaN,Did Not Participate In Practice,3.0,0.25,0.120
126,LA,Puka Nacua,WR,NaN,NaN,Did Not Participate In Practice,1.0,0.25,0.098
89,HOU,Nico Collins,WR,NaN,NaN,Did Not Participate In Practice,1.0,0.25,0.098
224,SF,Mike Evans,WR,NaN,NaN,Did Not Participate In Practice,1.0,0.25,0.098
25,BAL,Zay Flowers,WR,NaN,NaN,Did Not Participate In Practice,1.0,0.25,0.098


## Data Availability Warning

The injury layer is only as current as the official report.

If Week 2 reports are incomplete today, that is expected. Rerun this notebook later in the week before locking the final projection board.


In [10]:
if len(injuries) == 0:
    print(
        "WARNING: No official Week "
        f"{TARGET_WEEK} injury rows are available yet. "
        "Current injury adjustment is therefore zero. "
        "Rerun this notebook when reports are published."
    )
elif injuries["report_status"].isna().mean() > 0.75:
    print(
        "WARNING: Most players do not yet have final game-status "
        "designations. Treat these injury adjustments as preliminary "
        "and rerun closer to kickoff."
    )
else:
    print("Injury report contains substantial game status information.")


In [11]:
WEEKLY_DATA_DIR.mkdir(parents=True, exist_ok=True)

injury_adjusted.to_parquet(OUTPUT_PATH, index=False)
injury_detail.to_parquet(INJURY_DETAIL_PATH, index=False)

print("Saved:", OUTPUT_PATH)
print("Saved:", INJURY_DETAIL_PATH)


Saved: ..\..\data\processed\weekly\week_03_injury_adjusted_team_strength.parquet
Saved: ..\..\data\processed\weekly\week_03_injury_adjustments.parquet
